# Travelling Salesman Problem via QUBO + QAOA

**Find the shortest route through N cities using quantum computing and the Divi SDK**

---

The [Travelling Salesman Problem](https://en.wikipedia.org/wiki/Travelling_salesman_problem) (TSP) asks: *given a set of cities, what is the shortest route that visits every city exactly once and returns to the start?*

This notebook solves TSP by:
1. **Generate cities** — random 2D coordinates with Euclidean distances
2. **Build a QUBO** — encode the problem as n² binary variables using one-hot position encoding
3. **Classical baseline** — brute-force the optimal tour for comparison
4. **Solve with QAOA** — use Divi's QAOA to find the shortest tour
5. **Decode & repair** — extract valid tours from quantum bitstrings
6. **Compare** — visualise classical vs. quantum solutions side by side

## Setup & Imports

In [ ]:
import itertools

import dimod
import matplotlib.pyplot as plt
import numpy as np

from divi.backends import ParallelSimulator, QoroService, JobConfig
from divi.qprog.algorithms import QAOA
from divi.qprog.optimizers import MonteCarloOptimizer

### Choose Your Backend

Set `USE_CLOUD = True` to run on QoroService cloud backend instead of local simulation.

Get your API key at [dash.qoroquantum.net](https://dash.qoroquantum.net).

In [ ]:
USE_CLOUD = False  # Set to True to use QoroService cloud backend
SHOTS = 20_000

if USE_CLOUD:
    # QoroService automatically reads QORO_API_KEY from your environment
    backend = QoroService(config=JobConfig(shots=SHOTS))
    print("☁️  Using QoroService cloud backend")
else:
    backend = ParallelSimulator(shots=SHOTS)
    print("💻 Using local ParallelSimulator")

## Step 1 — Generate cities

We place `N_CITIES` cities randomly in the unit square and compute the Euclidean distance matrix.

| Parameter | Description |
|-----------|-------------|
| `N_CITIES` | Number of cities (keep ≤5 for local simulation — n² qubits!) |
| `SEED` | Random seed for reproducible city placement |

In [ ]:
N_CITIES = 4
SEED = 42


def generate_cities(n_cities, seed=42):
    """Generate random 2D city coordinates in the unit square."""
    rng = np.random.default_rng(seed)
    return rng.random((n_cities, 2))


def compute_distance_matrix(cities):
    """Compute the Euclidean distance matrix between all city pairs."""
    diff = cities[:, np.newaxis, :] - cities[np.newaxis, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=-1))


cities = generate_cities(N_CITIES, seed=SEED)
dist_matrix = compute_distance_matrix(cities)

print(f"📍 {N_CITIES} cities generated")
print(f"Distance matrix:\n{np.round(dist_matrix, 3)}")

### Visualise the cities

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(cities[:, 0], cities[:, 1], s=180, c="#4FC3F7",
           edgecolors="#0D47A1", zorder=5, linewidths=2)
for i, (x, y) in enumerate(cities):
    ax.annotate(str(i), (x, y), fontsize=11, ha="center", va="center",
                fontweight="bold", zorder=6)
ax.set_title(f"TSP — {N_CITIES} Cities", fontsize=14)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Step 2 — Build the QUBO

We encode TSP using the standard **one-hot position encoding**:

$$x_{i,t} = 1 \iff \text{city } i \text{ is at position } t \text{ in the tour}$$

This gives us **n² binary variables**. The QUBO energy function has three parts:

1. **Route cost** — minimise $\sum_{t} d(i,j) \cdot x_{i,t} \cdot x_{j,t+1}$
2. **One-city-per-step** — penalty $\lambda \cdot (\sum_i x_{i,t} - 1)^2$ for each time step
3. **One-step-per-city** — penalty $\lambda \cdot (\sum_t x_{i,t} - 1)^2$ for each city

The penalty weight $\lambda$ must be large enough to make constraint violations energetically unfavourable.

In [ ]:
def _var(city, step):
    """Variable name for 'city visited at step'."""
    return f"x_{city}_{step}"


def build_tsp_qubo(dist_matrix, penalty=None):
    """Build a BQM encoding the TSP as a QUBO."""
    n = len(dist_matrix)
    if penalty is None:
        penalty = 8.0 * np.sum(dist_matrix) / n

    bqm = dimod.BinaryQuadraticModel(vartype="BINARY")
    var_names = [_var(i, t) for i in range(n) for t in range(n)]

    # 1. Route cost
    for t in range(n):
        t_next = (t + 1) % n
        for i in range(n):
            for j in range(n):
                if i != j:
                    bqm.add_quadratic(_var(i, t), _var(j, t_next), dist_matrix[i, j])

    # 2. One-city-per-step constraint
    for t in range(n):
        bqm.offset += penalty
        for i in range(n):
            bqm.add_linear(_var(i, t), -penalty)
        for i in range(n):
            for j in range(i + 1, n):
                bqm.add_quadratic(_var(i, t), _var(j, t), 2 * penalty)

    # 3. One-step-per-city constraint
    for i in range(n):
        bqm.offset += penalty
        for t in range(n):
            bqm.add_linear(_var(i, t), -penalty)
        for t in range(n):
            for s in range(t + 1, n):
                bqm.add_quadratic(_var(i, t), _var(i, s), 2 * penalty)

    return bqm, var_names


bqm, var_names = build_tsp_qubo(dist_matrix)
print(f"Built QUBO: {len(var_names)} binary variables ({N_CITIES}² one-hot encoding)")
print(f"{len(bqm.quadratic)} quadratic interactions")

## Step 3 — Classical brute force (for comparison)

For small instances we can enumerate all $(n-1)!$ tours to find the exact optimum. This gives us a reference to evaluate the quantum solution against.

In [ ]:
def tour_length(tour, dist_matrix):
    """Total distance of a cyclic tour."""
    return sum(dist_matrix[tour[i], tour[(i + 1) % len(tour)]] for i in range(len(tour)))


def classical_brute_force(dist_matrix):
    """Find the shortest tour by exhaustive enumeration."""
    n = len(dist_matrix)
    best_tour, best_dist = None, float("inf")
    for perm in itertools.permutations(range(1, n)):
        candidate = [0] + list(perm)
        d = tour_length(candidate, dist_matrix)
        if d < best_dist:
            best_dist = d
            best_tour = candidate
    return best_tour, best_dist


classical_tour, classical_dist = classical_brute_force(dist_matrix)
print(f"Classical optimum: {classical_tour}")
print(f"Distance: {classical_dist:.4f}")

## Step 4 — Solve with QAOA

Divi's `QAOA` class accepts a `dimod.BinaryQuadraticModel` directly — no manual Hamiltonian construction needed.

We use the `MonteCarloOptimizer` to search for optimal QAOA angles.

In [ ]:
N_LAYERS = 3
MAX_ITERATIONS = 20

optimizer = MonteCarloOptimizer(population_size=50, n_best_sets=5)

qaoa = QAOA(
    problem=bqm,
    n_layers=N_LAYERS,
    max_iterations=MAX_ITERATIONS,
    optimizer=optimizer,
    backend=backend,
)

print(f"🚀 Running QAOA with {N_LAYERS} layers, {MAX_ITERATIONS} iterations...")
qaoa.run()
print("✅ Done!")

## Step 5 — Decode & repair quantum solutions

QAOA produces a probability distribution over bitstrings. We inspect the top candidates and:
1. **Exact decode** — check if the bitstring is a valid tour (all constraints satisfied)
2. **Greedy repair** — for near-feasible bitstrings, greedily assign cities to time slots based on activation values

In [ ]:
def decode_tour(sample, n_cities):
    """Decode a binary assignment into a tour, or None if infeasible."""
    tour = []
    for t in range(n_cities):
        cities_at_t = [i for i in range(n_cities) if sample.get(_var(i, t), 0) == 1]
        if len(cities_at_t) != 1:
            return None
        tour.append(cities_at_t[0])
    if len(set(tour)) != n_cities:
        return None
    return tour


def repair_tour(sample, n_cities):
    """Greedily repair an infeasible bitstring into a valid tour."""
    scores = np.zeros((n_cities, n_cities))
    for t in range(n_cities):
        for i in range(n_cities):
            scores[t][i] = sample.get(_var(i, t), 0)
    tour = []
    used = set()
    for t in range(n_cities):
        order = np.argsort(-scores[t])
        for i in order:
            if i not in used:
                tour.append(int(i))
                used.add(int(i))
                break
    remaining = set(range(n_cities)) - used
    tour.extend(sorted(remaining))
    return tour


# Scan top solutions
var_list = list(bqm.variables)
top_solutions = qaoa.get_top_solutions(n=100)
best = None
n_feasible = 0

for sol in top_solutions:
    sample = {var_list[k]: int(sol.bitstring[k]) for k in range(len(var_list))}
    tour = decode_tour(sample, N_CITIES)
    if tour is not None:
        n_feasible += 1
        d = tour_length(tour, dist_matrix)
        if best is None or d < best[1]:
            best = (tour, d)
    else:
        tour = repair_tour(sample, N_CITIES)
        d = tour_length(tour, dist_matrix)
        if best is None or d < best[1]:
            best = (tour, d)

quantum_tour, quantum_dist = best
print(f"Inspected {len(top_solutions)} bitstrings, {n_feasible} exactly feasible")
print(f"\n✅ Best tour: {quantum_tour}")
print(f"   Distance:  {quantum_dist:.4f}")

## Step 6 — Compare results

Side-by-side comparison of the classical brute-force optimum and the QAOA solution.

In [ ]:
ratio = quantum_dist / classical_dist

print("=" * 60)
print(f"  Classical:  tour = {classical_tour}   distance = {classical_dist:.4f}")
print(f"  Quantum:    tour = {quantum_tour}   distance = {quantum_dist:.4f}")
print("=" * 60)

if abs(ratio - 1.0) < 0.01:
    print("\n🎉 QAOA found the optimal tour!")
else:
    print(f"\n⚡ Quantum / Classical ratio = {ratio:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

for ax, t, d, label, col in [
    (ax1, classical_tour, classical_dist, "Classical (Brute Force)", "#FF7043"),
    (ax2, quantum_tour, quantum_dist, "Quantum (QAOA)", "#4FC3F7"),
]:
    for k in range(len(t)):
        i, j = t[k], t[(k + 1) % len(t)]
        ax.annotate(
            "", xy=cities[j], xytext=cities[i],
            arrowprops=dict(arrowstyle="->", color=col, lw=2),
        )
    ax.scatter(cities[:, 0], cities[:, 1], s=180, c="#E3F2FD",
               edgecolors="#0D47A1", zorder=5, linewidths=2)
    for i, (x, y) in enumerate(cities):
        ax.annotate(str(i), (x, y), fontsize=11, ha="center", va="center",
                    fontweight="bold", zorder=6)
    ax.set_title(f"{label}\ndistance = {d:.4f}", fontsize=12)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")

plt.suptitle("Travelling Salesman — Classical vs. Quantum", fontsize=14,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()